In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import math
from dataclasses import dataclass, field
from copy import deepcopy
import itertools
from operator import attrgetter as attrg
import functools
import cv2
import numpy as np

In [ ]:
DS = Path("../../../datasets")
BLUR_DS = DS / "blur"
PLASTIC_BAG = BLUR_DS / "imgs-with-distances" / "plastic-bag"

IMG_ROOT = PLASTIC_BAG / "10x4"

In [ ]:
import json
import csv
from dataclasses import asdict, fields


@dataclass
class BorderPixels:
    min: int
    max: int
    # this last one is subjective
    most_seen: int


@dataclass
class Result:
    src: Path
    target: Path
    src_dist_r: int
    src_dist_c: int
    target_dist_r: int
    target_dist_c: int
    src_bbox_h: int
    src_bbox_w: int
    target_bbox_h: int
    target_bbox_w: int
    src_horiz_border_pxls: BorderPixels
    src_vert_border_pxls: BorderPixels
    target_horiz_border_pxls: BorderPixels
    target_vert_border_pxls: BorderPixels
    blur_before_k: int
    blur_before_sigma: int
    blur_after_k: int
    blur_after_sigma: int
    comments: str


class ResultGather:
    def __init__(self, out):
        self.out = Path(out)
        self.out.parent.mkdir(exist_ok=True, parents=True)
        self.fieldnames = [f.name for f in fields(Result)]
        if self.out.exists():
            with open(self.out, newline="") as f:
                self.r = list(csv.DictReader(f))
        else:
            self.r = []

    def append(self, r: Result):
        row = asdict(r)
        self.r.append(row)
        write_header = not self.out.exists() or self.out.stat().st_size == 0
        with open(self.out, "a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=self.fieldnames)
            if write_header:
                writer.writeheader()
            writer.writerow(row)

In [ ]:
from typing import Literal


def resize_h(img, h, interpolation):
    new_height = h
    aspect_ratio = new_height / img.shape[0]
    new_width = int(img.shape[1] * aspect_ratio)
    return cv2.resize(img, (new_width, new_height), interpolation=interpolation)


def resize_w(img, w, interpolation):
    new_width = w
    aspect_ratio = new_width / img.shape[1]
    new_height = int(img.shape[0] * aspect_ratio)
    return cv2.resize(img, (new_width, new_height), interpolation=interpolation)


def _get_tlrb(img, bb):
    t, l = bb.y0, bb.x0
    b, r = img.shape[0] - bb.y1, img.shape[1] - bb.x1
    return t, l, r, b

def _resize_gemini(img, new_height, new_width, scale_factor):
    img_float = img.astype(np.float32) / 255.0
    img_linear = np.power(img_float, 2.2)

    # 2. Apply the "Physics" (Blur + Resize)
    # The sigma should be calculated to prevent aliasing based on the scale factor
    sigma = 0.5 * (1 / scale_factor) 
    blurred = cv2.GaussianBlur(img_linear, (0, 0), sigmaX=sigma)
    resized_linear = cv2.resize(blurred, (new_width, new_height), interpolation=cv2.INTER_AREA)

    # 3. Re-apply Gamma and convert back to 8-bit
    img_res_float = np.power(resized_linear, 1.0/2.2)
    return (np.clip(img_res_float, 0, 1) * 255).astype(np.uint8)


def resize_gemini_w(img, w, scale_factor):
    # 1. Convert to float and Linearize (Gamma 2.2)
    new_width = w
    aspect_ratio = new_width / img.shape[1]
    new_height = int(img.shape[0] * aspect_ratio)
    return _resize_gemini(img, new_height, new_width, scale_factor)

def resize_gemini_h(img, h, scale_factor):
    # 1. Convert to float and Linearize (Gamma 2.2)
    new_height = h
    aspect_ratio = new_height / img.shape[0]
    new_width = int(img.shape[1] * aspect_ratio)
    return _resize_gemini(img, new_height, new_width, scale_factor)


# The standard resize when run on the first example of hit bug sprays was not enough.
# The standard resize works on the entire image.
# So I’ve been taking random crops of images and passing them to resize.
# The new algorithm gets the bounding box of each object and
# finds the dimensions of the source object's crop that need to be resized
# to match the target crop's shape.
# For this, I would just find the resize ratio for the actual bounding boxes. We are using the width here for resizing.
# Then we find how much padding we need in the source box, which depends on the padding in the target image and the resizing padding.
# Once we have that, we get the crop that needs to be resized to the target image size.
# We resize that crop using standard CV2 resize, preserving the aspect ratio between the source and target crops.
def resize_with_bboxes(
    src, src_bbox, target, target_bbox, interpolation, resize_by: Literal["h", "w"]
):
    # the bounded box in src is resized to bounded box in target
    alpha = target_bbox.w / src_bbox.w
    t, l, r, b = _get_tlrb(target, target_bbox)
    t1, l1, r1, b1 = t / alpha, l / alpha, r / alpha, b / alpha

    y0 = int(max(src_bbox.y0 - t1, 0))
    x0 = int(max(src_bbox.x0 - l1, 0))
    y1 = int(min(src_bbox.y1 + b1, src.shape[0]))
    x1 = int(min(src_bbox.x1 + r1, src.shape[1]))

    crop_to_resize = src[y0:y1, x0:x1]
    print("will resize shape", crop_to_resize.shape, "to", target.shape)

    if resize_by == "w":
        return resize_gemini_w(crop_to_resize, target.shape[1], 1)
        # return resize_w(crop_to_resize, target.shape[1], interpolation)
    else:
        return resize_gemini_h(crop_to_resize, target.shape[0], 1)
        # return resize_h(crop_to_resize, target.shape[0], interpolation)


def add_luminosity_lab(rgb, add_constant):
    lab_rsz = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    lab_rsz[:, :, 0] = np.clip(
        lab_rsz[:, :, 0].astype(np.int16) + add_constant, 0, 255
    ).astype(np.uint8)
    return cv2.cvtColor(lab_rsz, cv2.COLOR_LAB2RGB)


def show(crops, figsize=None, ncols=2):
    crops = list(crops)
    crops = [c.rgb if isinstance(c, ImageAndDist) else c for c in crops]
    rows = math.ceil(len(crops) / ncols)
    if figsize is None:
        figsize = (10 * rows, 10 * rows)
        print("figsize", figsize)
    _, axs = plt.subplots(rows, ncols, figsize=figsize)
    axs = axs.flatten()
    for i, c in enumerate(crops):
        axs[i].imshow(c)
    plt.tight_layout()
    plt.show()


def sobel_edge_detection(img_rgb):
    # Load the image in grayscale
    # Compute the 1st order Sobel derivative in X-direction
    # ddepth is set to cv2.CV_64F to handle negative gradient values
    img = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    sobelx = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)

    # Compute the 1st order Sobel derivative in Y-direction
    sobely = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)

    # Compute the magnitude of the gradient (optional, for combined edge strength)
    # The gradient magnitude is calculated as sqrt(sobelx^2 + sobely^2)
    grad_magnitude = np.sqrt(sobelx**2 + sobely**2)

    # Convert results back to 8-bit image for display
    # Taking absolute values is important before converting to uint8
    return (
        cv2.convertScaleAbs(sobelx),
        cv2.convertScaleAbs(sobely),
        cv2.convertScaleAbs(grad_magnitude),
    )


def compose2(f, g):
    return lambda *a, **kw: f(g(*a, **kw))


def compose(*fs):
    return functools.reduce(compose2, fs)


def pipe(*fs):
    fs = reversed(fs)
    return functools.reduce(compose2, fs)


def mapl(f, lst):
    return list(map(f, lst))


def image_distance_sorter_key(img):
    return int(img.stem.split("x")[0])


def coord_from_path(path):
    comps = path.stem.split("x")
    y, x = comps[0], comps[1]
    return int(y), int(x)


@dataclass
class Coord:
    x: int
    y: int

    @classmethod
    def from_path(cls, path: Path) -> "Coord":
        comps = path.stem.split("x")
        y, x = comps[0], comps[1]
        return cls(y=int(y), x=int(x))


@dataclass
class BBox:
    y0: int
    y1: int
    x0: int
    x1: int

    @property
    def h(self):
        return self.y1 - self.y0

    @property
    def w(self):
        return self.x1 - self.x0


@dataclass
class ImageAndDist:
    us: Coord
    root: Coord
    path: Path
    rgb: np.ndarray

    @classmethod
    def from_path(cls, us, root, path):
        return ImageAndDist(us, root, path, plt.imread(path))

    def __getitem__(self, idx):
        if isinstance(idx, BBox):
            rgb = self.rgb[idx.y0 : idx.y1, idx.x0 : idx.x1]
        else:
            rgb = self.rgb[idx]
        return ImageAndDist(self.us, self.root, self.path, rgb)

    def resized_w(self, w, interpolation):
        # rgb = resize_w(self.rgb, w, interpolation)
        rgb = resize_gemini_w(self.rgb, w, 1)
        return ImageAndDist(self.us, self.root, self.path, rgb)

    def resized_h(self, h, interpolation):
        rgb = resize_gemini_h(self.rgb, h, 1)
        # rgb = resize_h(self.rgb, h, interpolation)
        return ImageAndDist(self.us, self.root, self.path, rgb)

    def resized_to_another(self, our_bbox, other, other_bbox, interpolation, resize_by):
        res = resize_with_bboxes(
            self.rgb, our_bbox, other.rgb, other_bbox, interpolation, resize_by
        )
        return self.clone_with_diff_rgb(self, res)

    @property
    def shape(self):
        return self.rgb.shape

    @classmethod
    def clone_with_diff_rgb(cls, src, rgb) -> "ImageAndDist":
        return cls(src.us, src.root, src.path, rgb)

    @classmethod
    def to_res_params_src(cls, src):
        return {
            "src_path": src.path,
            "src_dist_r": src.us.y,
            "src_dist_c": src.us.x,
        }

    @classmethod
    def to_res_params_target(cls, target):
        return {
            "target_path": target.path,
            "target_dist_r": target.us.y,
            "target_dist_c": target.us.x,
        }


def rgbs(image_and_dists):
    return map(attrg("rgb"), image_and_dists)

In [ ]:
ROOT = Coord.from_path(IMG_ROOT)
closeup = ImageAndDist.from_path(
    us=deepcopy(ROOT), root=deepcopy(ROOT), path=IMG_ROOT / "closeup.jpeg"
)
COL = "x4"
gather = ResultGather(f"./blur_results/{IMG_ROOT.stem}_{COL}.csv")
x4_images = IMG_ROOT.glob(f"*{COL}.jpeg")
x4_images = sorted(x4_images, key=image_distance_sorter_key)
x4_images = [
    ImageAndDist.from_path(root=ROOT, path=p, us=Coord.from_path(p)) for p in x4_images
]

# (14x4) -> (19x4)

In [ ]:
c = x4_images[0]
x = x4_images[1]
show([c, x], figsize=(10, 10))

In [ ]:
near = c[750:1150, 300:820]
near_bbox = BBox(y0=80, y1=320, x0=54, x1=418)
far = x[800:980, 480:720]
far_bbox = BBox(y0=20, y1=140, x0=10, x1=220)


show([far[far_bbox], near[near_bbox], far, near], (10, 10), ncols=2)

In [ ]:
near_rsz = near.resized_to_another(near_bbox, far, far_bbox, cv2.INTER_AREA, "h")
show([near_rsz, far])

In [ ]:
near_edge = near_rsz[30:-60, 155 + 25 : -10]
far_edge = far[30:-60, 153 + 25 : -35]

show([near_edge, far_edge])

In [ ]:
og_near_edge, og_far_edge = near_edge, far_edge

In [ ]:
src_horiz_border_pxls = BorderPixels(2, 3, 3)
src_vert_border_pxls = BorderPixels(2, 4, 3)
target_horiz_border_pxls = BorderPixels(4, 5, 4)
target_vert_border_pxls = BorderPixels(4, 6, 4)
blur_after_k = 0
blur_after_sigma = 0
blur_before_k = 0
blur_before_sigma = 0
comments = "Gaussian blur did not work. Blur after: smooths it instead of blurring. Looks unnatural. Blur before also mixes colors intead of providing a good change. Far image seems to have some light reflection on the edge also"

In [ ]:
res = Result(
    src_horiz_border_pxls=src_horiz_border_pxls,
    target_horiz_border_pxls=target_horiz_border_pxls,
    src_vert_border_pxls=src_vert_border_pxls,
    target_vert_border_pxls=target_vert_border_pxls,
    blur_after_k=blur_after_k,
    blur_after_sigma=blur_after_sigma,
    blur_before_k=blur_before_k,
    blur_before_sigma=blur_before_sigma,
    comments=comments,
    src_bbox_w=near_bbox.w,
    src_bbox_h=near_bbox.h,
    target_bbox_w=far_bbox.w,
    target_bbox_h=far_bbox.h,
    src=near.path,
    src_dist_r=near.us.y,
    src_dist_c=near.us.x,
    target=far.path,
    target_dist_r=far.us.y,
    target_dist_c=far.us.x,
)

In [ ]:
# gather.append(res)

In [ ]:
# try standard blur to see if they end up being the same
show([cv2.GaussianBlur(near_edge.rgb, (3, 3), 1), far_edge])

In [ ]:
def blur_before_resize(k, sigma):
    near_rgb = cv2.GaussianBlur(c.rgb[750:1150, 300:820], (k, k), sigma, sigmaY=sigma)
    near = ImageAndDist.clone_with_diff_rgb(c, near_rgb)
    near_bbox = BBox(y0=80, y1=320, x0=54, x1=418)
    far = x[800:980, 480:720]
    far_bbox = BBox(y0=20, y1=140, x0=10, x1=220)
    near_rsz = near.resized_to_another(near_bbox, far, far_bbox, cv2.INTER_AREA, "h")
    near_edge = near_rsz[30:-60, 155 + 25 : -10]
    far_edge = far[30:-60, 153 + 25 : -35]
    return near_edge, near


bbf_near_edge, bbf_near = blur_before_resize(5, 1)

show([near_edge, far_edge, bbf_near_edge], (10, 10), ncols=3)

In [ ]:
show([near, bbf_near, far], (15, 15), ncols=3)

I need to start making it more systematic. I've to measure something, otherwise it's not going to give me any credibal results.  
- Measure the number of horizontal pixels that denote a border (after resize)
- In the sheet, I also want to put the distance, along with height, width of the target and the initial object
- Gaussian blur before resizing
- Gaussian blur after resizing
- Luminosity change constant

```yaml
- src
- target
- src_dist
- target_dist
- src_bbox_h 
- src_bbox_w
- target_bbox_h 
- target_bbox_w
- horiz_border_size_pxl 
- vert_border_size_pxl 
- blur_before_k
- blur_before_sigma
- blur_after_k
- blur_after_sigma
```

# (19x4) -> (24x4)

In [ ]:
show(x4_images)

In [ ]:
c = x4_images[1]
x = x4_images[2]

show([c, x], figsize=(10, 10))

In [ ]:
near = c[700:1120, 380:820]
near_bbox = BBox(y0=120, y1=240, x0=110, x1=304)
far = x[700:800, 430:600]
far_bbox = BBox(y0=11, y1=85, x0=22, x1=148)


show([far[far_bbox], near[near_bbox], far, near], (10, 10), ncols=2)
# show([near[near_bbox], near, far], ncols=3)

In [ ]:
near_rsz = near.resized_to_another(near_bbox, far, far_bbox, cv2.INTER_LINEAR, "h")
near_rsz_area = near.resized_to_another(near_bbox, far, far_bbox, cv2.INTER_AREA, "h")
show([near_rsz, near, near_rsz, far])

In [ ]:
near_edge = near_rsz[20:75, 120:-10]
near_edge_area = near_rsz_area[20:75, 120:-10]
far_edge = far[20:75, 125:-10]

def _get_bbf_edge():
    near = c[700:1120, 380:820]
    near_rgb = cv2.GaussianBlur(near.rgb, (3,3), 1, sigmaY=1)
    near = ImageAndDist.clone_with_diff_rgb(near, near_rgb)
    near_bbox = BBox(y0=120, y1=240, x0=110, x1=304)
    far = x[700:800, 430:600]
    far_bbox = BBox(y0=11, y1=85, x0=22, x1=148)
    near_rsz = near.resized_to_another(near_bbox, far, far_bbox, cv2.INTER_LINEAR, "h")
    near_rsz_area = near.resized_to_another(near_bbox, far, far_bbox, cv2.INTER_AREA, "h")
    near_edge = near_rsz[20:75, 120:-10]
    near_edge_area = near_rsz_area[20:75, 120:-10]
    far_edge = far[20:75, 125:-10]
    return near_edge_area, near

bbf_near_edge, bbf_near = _get_bbf_edge()

show([
    near_edge_area, far_edge,
    cv2.GaussianBlur(near_edge_area.rgb, (3,3), 1, sigmaY=1), far_edge,
    bbf_near_edge, far_edge
], ncols=2)

In [ ]:
src_horiz_border_pxls = BorderPixels(3, 3, 3)
src_vert_border_pxls = BorderPixels(2, 5, 3)
target_horiz_border_pxls = BorderPixels(3, 6, 3)
target_vert_border_pxls = BorderPixels(4, 7, 4)
blur_after_k = 0
blur_after_sigma = 0
blur_before_k = 0
blur_before_sigma = 0
comments = "Gaussian blur seems to be a lost cause, resizing and blurring are smoothening the image a lot compared to the actual image in the distance"
res = Result(
    src_horiz_border_pxls=src_horiz_border_pxls,
    target_horiz_border_pxls=target_horiz_border_pxls,
    src_vert_border_pxls=src_vert_border_pxls,
    target_vert_border_pxls=target_vert_border_pxls,
    blur_after_k=blur_after_k,
    blur_after_sigma=blur_after_sigma,
    blur_before_k=blur_before_k,
    blur_before_sigma=blur_before_sigma,
    comments=comments,
    src_bbox_w=near_bbox.w,
    src_bbox_h=near_bbox.h,
    target_bbox_w=far_bbox.w,
    target_bbox_h=far_bbox.h,
    src=near.path,
    src_dist_r=near.us.y,
    src_dist_c=near.us.x,
    target=far.path,
    target_dist_r=far.us.y,
    target_dist_c=far.us.x,
)

In [ ]:
# gather.append(res)

# (24x4) -> (29x4)

In [ ]:
c = x4_images[2]
x = x4_images[3]

show([c, x], figsize=(10, 10))

In [ ]:

near = c[600:900, 350:680]
near_bbox = BBox(y0=111, y1=185, x0=102, x1=228)
far = x[730:825, 480:620]
far_bbox = BBox(y0=22, y1=72, x0=25, x1=115)

# show([far, near], (10, 10), ncols=2)
show([far[far_bbox], near[near_bbox], far, near], (10, 10), ncols=2)
# show([near[near_bbox], near, far], ncols=3)

In [ ]:

near_rsz = near.resized_to_another(near_bbox, far, far_bbox, cv2.INTER_AREA, "h")
show([near_rsz, near, near_rsz, far])

In [ ]:
near_edge = near_rsz[20:75, 90:-10]
far_edge = far[20:75, 90:-10]

# def _get_bbf_edge():
#     pass

# bbf_near_edge, bbf_near = _get_bbf_edge()

show([
    near_edge, far_edge,
    # cv2.GaussianBlur(near_edge.rgb, (3,3), 1, sigmaY=1), far_edge,
    # bbf_near_edge, far_edge
], ncols=2)

In [ ]:
src_horiz_border_pxls = BorderPixels(2, 2, 3)
src_vert_border_pxls = BorderPixels(3, 5, 3)
target_horiz_border_pxls = BorderPixels(2, 3, 3)
target_vert_border_pxls = BorderPixels(3, 5, 3)
blur_after_k = 0
blur_after_sigma = 0
blur_before_k = 0
blur_before_sigma = 0
comments = "These two look quite similar, it does seem that the resized version is less sharp again"
res = Result(
    src_horiz_border_pxls=src_horiz_border_pxls,
    target_horiz_border_pxls=target_horiz_border_pxls,
    src_vert_border_pxls=src_vert_border_pxls,
    target_vert_border_pxls=target_vert_border_pxls,
    blur_after_k=blur_after_k,
    blur_after_sigma=blur_after_sigma,
    blur_before_k=blur_before_k,
    blur_before_sigma=blur_before_sigma,
    comments=comments,
    src_bbox_w=near_bbox.w,
    src_bbox_h=near_bbox.h,
    target_bbox_w=far_bbox.w,
    target_bbox_h=far_bbox.h,
    src=near.path,
    src_dist_r=near.us.y,
    src_dist_c=near.us.x,
    target=far.path,
    target_dist_r=far.us.y,
    target_dist_c=far.us.x,
)

In [ ]:
# gather.append(res)

# (29x4) -> (34x4)

In [ ]:
c = x4_images[3]
x = x4_images[4]

show([x, c])


In [ ]:
near = c[650:880, 420:680]
near_bbox = BBox(y0=100, y1=154, x0=86, x1=174)
far = x[680:780, 500:630]
far_bbox = BBox(y0=32, y1=72, x0=33, x1=101)

# show([far, near], (10, 10), ncols=2)
# print(far[far_bbox].shape, near[near_bbox].shape)
show([far[far_bbox], near[near_bbox], far, near], (10, 10), ncols=2)
# show([near[near_bbox], near, far], ncols=3)

In [ ]:
near_rsz = near.resized_to_another(near_bbox, far, far_bbox, cv2.INTER_AREA, "h")
near_29x4_rsz_to_34x4 = near_rsz

show([far, near_rsz, far[far_bbox], near_rsz[far_bbox]])

In [ ]:
far_edge = far[35:70, 85:110]
near_edge = near_rsz[35:70, 85:110]

# def _get_bbf_edge():
#     pass

# bbf_near_edge, bbf_near = _get_bbf_edge()

show([
    far_edge, near_edge,
    # cv2.GaussianBlur(near_edge.rgb, (3,3), 1, sigmaY=1), far_edge,
    # bbf_near_edge, far_edge
], ncols=2)

In [ ]:
src_horiz_border_pxls = BorderPixels(2, 4, 3)
src_vert_border_pxls = BorderPixels(3, 5, 3)
target_horiz_border_pxls = BorderPixels(2, 3, 3)
target_vert_border_pxls = BorderPixels(3, 5, 3)
blur_after_k = 0
blur_after_sigma = 0
blur_before_k = 0
blur_before_sigma = 0
comments = "same as before, the resized area is less sharp than original. in the original, edges are more pronounced, brighter are brighter, darker are darker"
res = Result(
    src_horiz_border_pxls=src_horiz_border_pxls,
    target_horiz_border_pxls=target_horiz_border_pxls,
    src_vert_border_pxls=src_vert_border_pxls,
    target_vert_border_pxls=target_vert_border_pxls,
    blur_after_k=blur_after_k,
    blur_after_sigma=blur_after_sigma,
    blur_before_k=blur_before_k,
    blur_before_sigma=blur_before_sigma,
    comments=comments,
    src_bbox_w=near_bbox.w,
    src_bbox_h=near_bbox.h,
    target_bbox_w=far_bbox.w,
    target_bbox_h=far_bbox.h,
    src=near.path,
    src_dist_r=near.us.y,
    src_dist_c=near.us.x,
    target=far.path,
    target_dist_r=far.us.y,
    target_dist_c=far.us.x,
)

In [ ]:
# gather.append(res)

# (19x4) -> (34x4)

In [ ]:
c = x4_images[1]
x = x4_images[4]

show([x, c])

In [ ]:
near = c[600:1200, 300:900]
near_bbox = BBox(y0=220, y1=340, x0=187, x1=384)
far = x[680:780, 500:630]
far_bbox = BBox(y0=32, y1=72, x0=33, x1=101)

# show([far, near], (10, 10), ncols=2)
# print(far[far_bbox].shape, near[near_bbox].shape)
show([far[far_bbox], near[near_bbox], far, near], (10, 10), ncols=2)
# show([near[near_bbox], near, far], ncols=3)

In [ ]:

near_rsz = near.resized_to_another(near_bbox, far, far_bbox, cv2.INTER_AREA, "h")

show([far, near_rsz, far[far_bbox], near_rsz[far_bbox]])

In [ ]:
show([near_29x4_rsz_to_34x4, far, near_rsz], (15,15), ncols=3)

In [ ]:
far_edge = far[35:70, 85:110]
near_edge = near_rsz[35:70, 85:110]

# def _get_bbf_edge():
#     pass

# bbf_near_edge, bbf_near = _get_bbf_edge()

show([
    far_edge, near_edge,
    # cv2.GaussianBlur(near_edge.rgb, (3,3), 1, sigmaY=1), far_edge,
    # bbf_near_edge, far_edge
], ncols=2)


# Next steps

Counting pixels isn’t very objective right now because I can’t discern what a border is after a single point.  
Although I can see it at a higher level, the border itself gets hazy when you look down on it. When you look closely, it’s difficult to tell whether a pixel belongs to the border.  

The method was to take photos of the same object at different distances. We are trying to translate the object from a closer distance to a farther distance. The difference between the resized closer object and the original farther object is that the resized version is less sharp than the original farther version.   

The number of pixels that compose the border is less, I'm guessing. I have to test it. A good, faithful algorithm for downscaling might be a better resizing algorithm instead of using different blur types.  

The resize algorithm gets worse the more your resize jump is. We need to measure the sharpness of the image somehow (how much the lines defining "edges" are visible)